# Exploratory Data Analysis - Instacart Dataset

This notebook performs a comprehensive exploratory analysis of the bronze layer tables to understand:
- **Dataset scale and structure**
- **Customer ordering behavior and patterns**
- **Product catalog characteristics**
- **Temporal trends** (when do customers order?)
- **Reorder behavior** (customer loyalty indicators)
- **Department and product performance**

The insights will guide feature engineering and model development for predicting customer purchase behavior.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [0]:
bronze_schema = "big_data.bronze"

print(f"Analyzing tables from: {bronze_schema}")

In [0]:
# Get overview of all bronze tables
print(f"Tables in {bronze_schema}:\n")
spark.sql(f"SHOW TABLES IN {bronze_schema}").show()

print("\nTable sizes:")
tables = ["departments", "aisles", "products", "orders", "order_products_prior", "order_products_train"]
for table in tables:
    count = spark.table(f"{bronze_schema}.{table}").count()
    print(f"  {table}: {count:,} rows")

In [0]:
orders = spark.table(f"{bronze_schema}.orders")

print("Orders Schema:")
orders.printSchema()

print("\nSample data:")
display(orders.limit(5))

print(f"\nTotal orders: {orders.count():,}")
print(f"Unique users: {orders.select('user_id').distinct().count():,}")

## Orders Analysis - Key Insights

**Dataset Overview:**
- Contains 3.4M+ orders from 206K+ unique users
- Average of ~16.6 orders per user
- Includes timing information (day of week, hour of day)
- Tracks order sequence and days between orders

**Why this matters:**
- Understanding order frequency helps predict next purchase timing
- User segmentation can be based on ordering patterns
- Temporal features will be important for recommendation models

## Temporal Patterns - Key Insights

**Peak Ordering Times:**
- **Sunday (day 0) and Monday (day 1)** are the busiest days
- **Peak hours: 10am-3pm** - mid-morning to early afternoon
- Early morning (midnight-6am) has minimal activity

**Business Implications:**
- Schedule promotions and marketing for Sunday/Monday mornings
- Inventory planning should account for weekend demand spikes
- Server capacity planning for peak hours
- Time-based features (hour, day of week) will be strong predictors

## User Ordering Behavior - Key Insights

**Order Frequency:**
- Average ~17 orders per user (ranges from 4 to 100)
- Average ~17 days between orders
- Some users order weekly, others monthly

**User Segmentation Opportunities:**
- **High-frequency users** (weekly orders) - loyalty program candidates
- **Medium-frequency users** (bi-weekly) - typical customers
- **Low-frequency users** (monthly+) - re-engagement targets

**Model Implications:**
- Days since prior order is a strong feature
- User-level aggregations (lifetime orders, avg frequency) will help personalization

In [0]:
# Analyze order timing patterns
orders_temporal = orders.groupBy("order_dow", "order_hour_of_day") \
    .agg(F.count("*").alias("order_count")) \
    .orderBy("order_dow", "order_hour_of_day")

print("Orders by day of week and hour:")
display(orders_temporal)

# Most popular ordering times
print("\nTop 10 busiest ordering times:")
display(orders_temporal.orderBy(F.desc("order_count")).limit(10))

In [0]:
# Analyze user ordering patterns
user_stats = orders.groupBy("user_id").agg(
    F.count("*").alias("total_orders"),
    F.max("order_number").alias("max_order_number"),
    F.avg("days_since_prior_order").alias("avg_days_between_orders")
)

print("User ordering behavior:")
user_stats.select(
    F.mean("total_orders").alias("avg_orders_per_user"),
    F.min("total_orders").alias("min_orders"),
    F.max("total_orders").alias("max_orders"),
    F.mean("avg_days_between_orders").alias("avg_days_between")
).show()

print("\nOrder frequency distribution:")
user_stats.groupBy("total_orders").count() \
    .orderBy("total_orders") \
    .limit(20) \
    .show()

In [0]:
products = spark.table(f"{bronze_schema}.products")

print("Products Schema:")
products.printSchema()

print("\nSample products:")
display(products.limit(10))

print(f"\nTotal products: {products.count():,}")
print(f"Unique aisles: {products.select('aisle_id').distinct().count():,}")
print(f"Unique departments: {products.select('department_id').distinct().count():,}")

## Product Catalog - Key Insights

**Catalog Size:**
- **49,688 products** across 134 aisles and 21 departments
- Average ~370 products per aisle
- Average ~2,366 products per department

**Catalog Complexity:**
- Large SKU count indicates diverse product offerings
- Hierarchical structure: Department → Aisle → Product
- Product names contain descriptive information (brand, size, flavor)

**Modeling Implications:**
- Product embeddings can capture similar items
- Aisle and department groupings enable category-level recommendations
- Cold-start problem for new products with no history

## Product Distribution by Department - Key Insights

**Top Departments (by product count):**
1. **Personal Care** (6,563 products) - highest variety
2. **Snacks** (6,264 products)
3. **Pantry** (5,370 products)

**Observations:**
- Personal care and snacks have the most SKU diversity
- Essential categories (produce, dairy) have fewer SKUs but likely higher volume
- Bulk department has minimal products (38) - specialty category

**Strategy:**
- High-variety departments need stronger recommendation systems
- Essential departments may drive overall basket composition

In [0]:
departments = spark.table(f"{bronze_schema}.departments")
aisles = spark.table(f"{bronze_schema}.aisles")

print("Departments:")
display(departments.orderBy("department_id"))

print(f"\nTotal departments: {departments.count()}")

print("\nAisles (sample):")
display(aisles.orderBy("aisle_id").limit(20))

print(f"\nTotal aisles: {aisles.count()}")

In [0]:
# Analyze product distribution across departments
product_dept_dist = products.join(departments, "department_id") \
    .groupBy("department") \
    .agg(F.count("*").alias("product_count")) \
    .orderBy(F.desc("product_count"))

print("Products per department:")
display(product_dept_dist)

# Convert to pandas for visualization
pd_dept = product_dept_dist.toPandas()

plt.figure(figsize=(12, 6))
plt.barh(pd_dept['department'], pd_dept['product_count'])
plt.xlabel('Number of Products')
plt.ylabel('Department')
plt.title('Product Distribution by Department')
plt.tight_layout()
plt.show()

In [0]:
# Combine prior and train datasets
order_products_prior = spark.table(f"{bronze_schema}.order_products_prior")
order_products_train = spark.table(f"{bronze_schema}.order_products_train")

order_products = order_products_prior.withColumn("dataset", F.lit("prior")) \
    .unionByName(order_products_train.withColumn("dataset", F.lit("train")))

print("Order Products Schema:")
order_products.printSchema()

print("\nDataset split:")
order_products.groupBy("dataset").count().show()

print(f"\nTotal order-product records: {order_products.count():,}")
print(f"Unique products ordered: {order_products.select('product_id').distinct().count():,}")
print(f"Unique orders: {order_products.select('order_id').distinct().count():,}")

## Order-Products Dataset - Key Insights

**Dataset Split:**
- **Prior orders**: 32.4M records (training data)
- **Train orders**: 1.4M records (evaluation set)
- Combined: 33.8M order-product transactions

**Coverage:**
- 49,685 unique products ordered (99.9% of catalog)
- 3.4M unique orders
- Average 10.1 items per order

**Why this matters:**
- This is the core transactional data for building recommendation models
- Links orders to products with reorder flags
- Cart position (add_to_cart_order) indicates product importance

## Reorder Behavior - Key Insights

**Overall Reorder Rate: 59%**
- Nearly 6 out of 10 product purchases are reorders
- Indicates **strong customer loyalty** and habitual purchasing
- Customers stick to familiar products

**Top Reordered Products:**
- Tend to be staples: bananas, organic produce, milk, water
- Essentials with high repurchase frequency
- Low price, high consumption rate items

**Modeling Opportunity:**
- Reorder prediction is a key use case
- Historical purchase data is highly predictive
- Focus on identifying which products users will reorder in next basket

## Cart Position Analysis - Key Insights

**Average Cart Size: ~10 items**
- First items added tend to have higher reorder rates (60-65%)
- Later items (position 20+) have lower reorder rates (45-50%)
- Suggests customers add essentials first, then explore

**Cart Building Behavior:**
- **Positions 1-5**: Core essentials (high reorder)
- **Positions 6-15**: Regular purchases
- **Positions 15+**: Exploratory/impulse items

**Model Feature:**
- Previous cart position can indicate product priority
- Products consistently added early are high-priority reorder candidates

## Department Performance - Key Insights

**Top Performing Departments (by volume):**
1. **Produce**: 9.9M items, 65% reorder rate, 3.95 items/order
2. **Dairy Eggs**: 5.6M items, **67% reorder rate** (highest!), 2.49 items/order
3. **Snacks**: 3.0M items, 57% reorder rate, 2.08 items/order

**Key Observations:**
- **Produce dominates** in volume - fresh items drive orders
- **Dairy eggs has highest loyalty** - essentials with top reorder rate
- **Personal care** has most SKUs (6,563) but lower volume (468K items)
- **Items per order** shows basket composition patterns

**Strategic Insights:**
- Focus on produce/dairy for retention strategies
- Personal care needs better discovery (high SKUs, low penetration)
- High reorder rates indicate opportunity for subscription models

In [0]:
# Analyze reorder behavior
reorder_stats = order_products.groupBy("reordered").count() \
    .withColumn("percentage", F.round(F.col("count") * 100 / order_products.count(), 2))

print("Reorder statistics:")
display(reorder_stats)

# Top reordered products
top_reordered = order_products.filter(F.col("reordered") == 1) \
    .groupBy("product_id") \
    .agg(F.count("*").alias("reorder_count")) \
    .join(products.select("product_id", "product_name"), "product_id") \
    .orderBy(F.desc("reorder_count")) \
    .limit(20)

print("\nTop 20 most reordered products:")
display(top_reordered)

In [0]:
# Analyze add_to_cart_order patterns
cart_stats = order_products.groupBy("add_to_cart_order").agg(
    F.count("*").alias("frequency"),
    F.avg(F.col("reordered").cast("int")).alias("avg_reorder_rate")
).orderBy("add_to_cart_order").limit(30)

print("Cart position analysis (first 30 positions):")
display(cart_stats)

print("\nAverage cart size:")
avg_cart_size = order_products.groupBy("order_id") \
    .agg(F.max("add_to_cart_order").alias("cart_size")) \
    .select(F.avg("cart_size").alias("avg_cart_size"))
    
display(avg_cart_size)

In [0]:
# Most frequently ordered products
popular_products = order_products.groupBy("product_id") \
    .agg(
        F.count("*").alias("order_frequency"),
        F.avg(F.col("reordered").cast("int")).alias("reorder_rate")
    ) \
    .join(products.select("product_id", "product_name", "aisle_id", "department_id"), "product_id") \
    .join(departments.select("department_id", "department"), "department_id") \
    .join(aisles.select("aisle_id", "aisle"), "aisle_id") \
    .select("product_name", "department", "aisle", "order_frequency", "reorder_rate") \
    .orderBy(F.desc("order_frequency")) \
    .limit(50)

print("Top 50 most popular products:")
display(popular_products)

In [0]:
# Department-level metrics
dept_performance = order_products \
    .join(products.select("product_id", "department_id"), "product_id") \
    .join(departments.select("department_id", "department"), "department_id") \
    .groupBy("department") \
    .agg(
        F.countDistinct("order_id").alias("unique_orders"),
        F.count("*").alias("total_items"),
        F.avg(F.col("reordered").cast("int")).alias("avg_reorder_rate"),
        F.countDistinct("product_id").alias("unique_products")
    ) \
    .withColumn("items_per_order", F.round(F.col("total_items") / F.col("unique_orders"), 2)) \
    .orderBy(F.desc("total_items"))

print("Department performance metrics:")
display(dept_performance)

In [0]:
print("="*80)
print("KEY INSIGHTS FROM EXPLORATORY ANALYSIS")
print("="*80)

# Dataset size
total_users = orders.select("user_id").distinct().count()
total_orders = orders.count()
total_products = products.count()
total_transactions = order_products.count()

print(f"\n1. DATASET SCALE:")
print(f"   - Total users: {total_users:,}")
print(f"   - Total orders: {total_orders:,}")
print(f"   - Total products: {total_products:,}")
print(f"   - Total order-product transactions: {total_transactions:,}")

# Reorder insights
reorder_rate = order_products.filter(F.col("reordered") == 1).count() / order_products.count() * 100
print(f"\n2. REORDER BEHAVIOR:")
print(f"   - Overall reorder rate: {reorder_rate:.1f}%")
print(f"   - High reorder rate indicates strong customer loyalty")

# Temporal patterns
peak_hour = orders.groupBy("order_hour_of_day").count() \
    .orderBy(F.desc("count")).first()["order_hour_of_day"]
peak_day = orders.groupBy("order_dow").count() \
    .orderBy(F.desc("count")).first()["order_dow"]

print(f"\n3. TEMPORAL PATTERNS:")
print(f"   - Peak ordering hour: {peak_hour}:00 (likely {peak_hour}:00-{peak_hour+1}:00)")
print(f"   - Peak ordering day of week: {peak_day} (0=Sunday, 6=Saturday)")

# Product diversity
avg_cart = order_products.groupBy("order_id").count() \
    .select(F.avg("count")).first()[0]

print(f"\n4. SHOPPING BEHAVIOR:")
print(f"   - Average items per order: {avg_cart:.1f}")
print(f"   - Indicates typical basket size")

print(f"\n5. DATA QUALITY:")
print(f"   - All tables successfully loaded from bronze layer")
print(f"   - Ready for silver layer transformations")

print("\n" + "="*80)

## Analysis Summary and Next Steps

### Key Findings:

**1. Strong Customer Loyalty**
- 59% reorder rate indicates habitual purchasing behavior
- Perfect opportunity for predictive reorder models

**2. Clear Temporal Patterns**
- Sunday/Monday 10am-3pm peak ordering times
- Can optimize promotions and inventory

**3. Department Dynamics**
- Produce and Dairy drive volume and loyalty
- Personal care has high variety but low penetration

**4. User Segmentation Potential**
- Wide range of order frequencies (4-100 orders)
- Average 17 days between orders

### Recommended Next Steps:

**Silver Layer Transformations:**
- Create user-level features (lifetime orders, avg frequency, favorite departments)
- Build product features (reorder rate, popularity score, department metrics)
- Engineer temporal features (day of week, hour, days since last order)

**Gold Layer Analytics:**
- User segmentation (high/medium/low frequency)
- Product affinity analysis (frequently bought together)
- Reorder prediction modeling
- Basket composition analysis

**Machine Learning Opportunities:**
- Next basket prediction
- Reorder probability scoring
- Personalized product recommendations
- Time-to-next-order prediction